## Determining the optimal number of hidden layers and neurons for an ANN

In [1]:
%pip install keras
%pip install scikeras


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import pickle
import tensorflow as tf
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from scikeras.wrappers import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = pd.read_csv(r'C:\Users\jenis\OneDrive\Desktop\TrainOfThought\TrainOfThought\Deep_Learning\annclassification\Churn_Modelling.csv')

In [4]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [6]:
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

In [7]:
ohe = OneHotEncoder(sparse_output=False)  # sparse_output replaces deprecated sparse=
geo_encoded = ohe.fit_transform(data[['Geography']])
geo_df = pd.DataFrame(geo_encoded, columns=ohe.get_feature_names_out(['Geography']), index=data.index)
data = pd.concat([data.drop('Geography', axis=1), geo_df], axis=1)

In [8]:
X = data.drop('Exited', axis=1)
y = data['Exited']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [11]:
# convert it into pickle file
with open('le.pkl',     'wb') as f: pickle.dump(le,     f)
with open('ohe.pkl',    'wb') as f: pickle.dump(ohe,    f)
with open('scaler.pkl', 'wb') as f: pickle.dump(scaler, f)


In [12]:
def create_model(neurons=32, layers=1, meta=None):
    # scikeras passes feature count via meta dict
    input_dim = meta["n_features_in_"] if meta else X_train.shape[1]
    model = Sequential()
    model.add(tf.keras.Input(shape=(input_dim,)))
    for _ in range(layers):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [13]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [14]:
# create a keras classifier

model = KerasClassifier(
    model=create_model,
    epochs=50,
    batch_size=10,
    callbacks=[early_stop],  
    validation_split=0.1,     # needed so val_loss is available for EarlyStopping
    verbose=0,
)

In [15]:
# grid search parameters
param_grid = {
    'model__neurons': [16, 32, 64, 128],
    'model__layers':  [1, 2],
    'epochs':         [50, 100],
}

In [16]:
# perform grid search
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=1,
    scoring='accuracy',
)

In [17]:
grid_result = grid.fit(X_train, y_train)

In [18]:
print("\nGrid Search Completed!")
print("Best CV Accuracy : {:.4f}".format(grid_result.best_score_))
print("Best Parameters  :", grid_result.best_params_)


Grid Search Completed!
Best CV Accuracy : 0.8586
Best Parameters  : {'epochs': 100, 'model__layers': 1, 'model__neurons': 64}
